In [672]:
import warnings
warnings.filterwarnings("ignore")

In [673]:
import pandas as pd
import xml.etree.ElementTree as et

# **Data processing**

**XML Parsing**

In [674]:
sentiment_label_encode = {
    'NONE': 0,
    'P': 1,
    'N': 2,
    'NEU': 3
}

In [675]:
def xml_2_dataframe(path):
    xtree = et.parse(path)
    xroot = xtree.getroot()

    df = pd.DataFrame({
        'id': pd.Series(dtype='str'),
        'label': pd.Series(dtype='str'),
        'text': pd.Series(dtype='str')
    })

    for tweet in xroot:
        tweet_id = tweet.find('tweetid').text
        content = tweet.find('content').text
        sentiment = tweet.find('sentiment').find('polarity').find('value').text
        #new_input = {'id':tweet_id, 'label': sentiment, 'text': content}
        #df = df.append(new_input, ignore_index=True)
        
        new_input = [tweet_id, sentiment, content]
        df.loc[len(df.index)] = new_input 

    return df

In [676]:
train_df = xml_2_dataframe('data/TASS2017_T1_training.xml')
dev_df = xml_2_dataframe('data/TASS2017_T1_development.xml')
test_df = xml_2_dataframe('data/TASS2017_T1_test.xml')

In [677]:
print(train_df.head())
print(dev_df.head())
print(test_df.head())

                   id label                                               text
0  768213876278165504  NONE  -Me caes muy bien \n-Tienes que jugar más part...
1  768213567418036224     N  @myendlesshazza a. que puto mal escribo\n\nb. ...
2  768212591105703936     N  @estherct209 jajajaja la tuya y la d mucha gen...
3  768221670255493120     P  Quiero mogollón a @AlbaBenito99 pero sobretodo...
4  768221021300264964     N  Vale he visto la tia bebiendose su regla y me ...
                   id label                                               text
0  770976639173951488     P  @noseashetero 1000/10 de verdad a ti que voy a...
1  771092421866389508     P  @piscolabisaereo @HistoriaNG @SPosteguillo las...
2  771092111429083136     P  Al final han sido 3h  Bueno, mañana tengo fies...
3  771092070572449796     N  @Jorge_Ruiz14 yo no tengo tiempo para esas cos...
4  771094192508600320     N  @_MissChaotic_ ves ese brillo? es un coso que ...
                   id label                         

**Tokenizer**

In [678]:
from nltk.tokenize import TweetTokenizer

def tokenize(dataframe):
    dataframe['tokenized_text'] = dataframe['text'].map(
        TweetTokenizer(strip_handles=False, reduce_len=True, preserve_case=False).tokenize
    )

    dataframe['tokenized_text'] = dataframe['tokenized_text'].map(
        ' '.join
    )

    return dataframe

In [679]:
train_df = tokenize(train_df)
dev_df = tokenize(dev_df)
test_df = tokenize(test_df)

In [680]:
print(train_df.shape)
print(dev_df.shape)
print(test_df.shape)

(1008, 4)
(506, 4)
(1899, 4)


In [681]:
import re

def vocab_reducer(text):
    res = []
    for word in text.split():
        word = re.sub('@.*','mención', word)
        word = re.sub('#(.*)', 'etiqueta', word)
        word = re.sub('http.*', 'web', word)
        word = re.sub('\d.*', 'número', word)
        res.append(word)
    return (res)


**Vectorizer**

In [682]:
from sklearn.feature_extraction.text import CountVectorizer, HashingVectorizer, TfidfTransformer, TfidfVectorizer
vectorizer = CountVectorizer(tokenizer=vocab_reducer, ngram_range=(1,4))
#transformer = TfidfTransformer(smooth_idf=True)
vectorizer = vectorizer.fit(train_df['tokenized_text'])
#sp_train = vectorizer.transform(train_df['tokenized_text'])
#transformer = transformer.fit(sp_train)

In [683]:
def vectorize(data):
    sp_matrix = vectorizer.transform(data['tokenized_text'])
    return sp_matrix #transformer.transform(sp_matrix)

In [684]:
train_vec = vectorize(train_df)
dev_vec = vectorize(dev_df)
test_vec = vectorize(test_df)

In [685]:
print(train_vec.shape)
print(dev_vec.shape)
print(test_vec.shape)

(1008, 43625)
(506, 43625)
(1899, 43625)


**Polarization count**

In [686]:
from nltk.stem import SnowballStemmer
spanish_stemmer = SnowballStemmer('spanish')

def stemmer(word):
    return spanish_stemmer.stem(word)

In [687]:
 
def load_polarity_lexicon(path):
    polarity_dict = {}
    with open(path,'r') as file:
        text = file.read()
        text = re.sub(r'#.*', '', text)
        for line in text.split('\n'):
            if len(line) >  0:
                word = re.split(r'(\t|\s+)',line)[0]
                tag = re.split(r'(\t|\s+)',line)[-1]
                polarity_dict[stemmer(word)] = tag
    
    return polarity_dict

In [688]:
polarity_dict = load_polarity_lexicon('./data/ElhPolar_esV1.lex')

In [689]:
print(polarity_dict.keys())

dict_keys(['a_cieg', 'a_flot', 'a_la_der', 'a_la_mod', 'a_la_sombr', 'a_pesar_d', 'a_salv', 'abandon', 'abarat', 'abat', 'abdic', 'aberraciã³n', 'abofet', 'abogar_por', 'abomin', 'abominaciã³n', 'abord', 'aborrec', 'abras', 'abraz', 'abrum', 'absolv', 'absorbent', 'absurd', 'abuch', 'abuche', 'abund', 'aburr', 'abusar_d', 'abus', 'abyect', 'acalor', 'acat', 'acced', 'acept', 'aceptaciã³n', 'acert', 'achaqu', 'aciag', 'aclamaciã³n', 'aclam', 'aclar', 'acobard', 'acogedor', 'acog', 'acoger_con_agr', 'acomod', 'acongoj', 'aconsej', 'acord', 'acort', 'acos', 'acritud', 'activ', 'acuchill', 'acuerd', 'acusaciã³n', 'acus', 'adapt', 'adecu', 'adherent', 'adher', 'adhesiã³n', 'adicciã³n', 'adict', 'adivin', 'adjudic', 'admir', 'admiraciã³n', 'admit', 'adoctrin', 'ador', 'adoraciã³n', 'adorn', 'adulaciã³n', 'adul', 'adulteraciã³n', 'adulter', 'adulterar_con_drog', 'adversari', 'advers', 'advertent', 'afabil', 'afabl', 'afan', 'afect', 'afectu', 'aficiã³n', 'afin', 'afirmaciã³n', 'afirm', 'aflic

In [690]:
import numpy as np

def polarity_count(dataframe):
    res = np.zeros((dataframe['tokenized_text'].size, 2))
    for i, sentence in enumerate(dataframe['tokenized_text']):
        for word in sentence.split(' '):
            polarity = polarity_dict.get(stemmer(word), None)
            if polarity == None:
                continue
            if polarity == 'positive':
                res[i, 0] += 1
            if polarity == 'negative':
                res[i, 1] += 1
    return res

In [691]:
train_pol_mat = polarity_count(train_df)
dev_pol_mat = polarity_count(dev_df)
test_pol_mat = polarity_count(test_df)

In [692]:
print(train_pol_mat.shape)
print(dev_pol_mat.shape)
print(test_pol_mat.shape)

(1008, 2)
(506, 2)
(1899, 2)


**Join feature vector and polarization count**

In [693]:
import scipy

train_matrix = scipy.sparse.hstack((train_vec, train_pol_mat))
dev_matrix = scipy.sparse.hstack((dev_vec, dev_pol_mat))
test_matrix = scipy.sparse.hstack((test_vec, test_pol_mat))

In [694]:
print(train_matrix.shape)
print(dev_matrix.shape)
print(test_matrix.shape)

(1008, 43627)
(506, 43627)
(1899, 43627)


In [695]:
#from sklearn import svm
#from sklearn.metrics import classification_report
#
#svm_classif = svm.LinearSVC(C=0.05)
#svm_classif.fit(train_matrix, train_df['label'])
#preds = svm_classif.predict(dev_matrix)
#
#print('accuracy: {}\n'.format(sum(preds==dev_df['label'])/len(dev_df['label'])))
#
#print(classification_report(dev_df['label'], preds))

**Train and eval system**

In [696]:
from sklearn import svm, tree
import sklearn.linear_model as lm
from sklearn.metrics import classification_report

def train_and_eval(classifier, kwargs):
    model = None

    if classifier == 'svc':
        model = svm.LinearSVC(**kwargs)
    if classifier == 'sgd':
        model = lm.SGDClassifier(**kwargs)
    if classifier == 'lr':
        model = lm.LogisticRegression(**kwargs)
    if classifier == 'perceptron':
        model = lm.Perceptron(**kwargs)
    if classifier == 'ridge':
        model = lm.RidgeClassifier(**kwargs)
    if classifier == 'tree':
        model = tree.DecisionTreeClassifier(**kwargs)

    model = model.fit(train_matrix, train_df['label'])

    preds = model.predict(dev_matrix)

    return classification_report(dev_df['label'], preds)

In [697]:
print(train_and_eval('svc', {'C':0.1}))

              precision    recall  f1-score   support

           N       0.61      0.77      0.68       219
         NEU       0.22      0.06      0.09        69
        NONE       0.38      0.21      0.27        62
           P       0.61      0.68      0.64       156

    accuracy                           0.58       506
   macro avg       0.45      0.43      0.42       506
weighted avg       0.53      0.58      0.54       506



# **Test Automation**

In [698]:
model_dict = {
    'svc':[
        #({'C':1, 'max_iter' : 7500},), 
        #({'C':0.5, 'max_iter' : 7500},), 
        ({'C':0.1, 'max_iter' : 10000},), 
        #({'C':0.05, 'max_iter' : 7500},), 
        #({'C':0.01, 'max_iter' : 7500},),
        #({'C':0.005, 'max_iter' : 7500},)
    ],
    'sgd':[
        #({'loss':'log_loss', 'alpha': 0.0001, 'max_iter' : 7500},),
        #({'loss':'log_loss', 'alpha': 0.0005, 'max_iter' : 7500},),
        #({'loss':'log_loss', 'alpha': 0.001, 'max_iter' : 7500},),
        #({'loss':'log_loss', 'alpha': 0.005, 'max_iter' : 7500},),
        #({'loss':'log_loss', 'alpha': 0.0005, 'penalty': 'elasticnet', 'max_iter' : 7500},),
        #({'loss':'log_loss', 'alpha': 0.001, 'penalty': 'elasticnet', 'max_iter' : 7500},),
        #({'loss':'log_loss', 'alpha': 0.005, 'penalty': 'elasticnet', 'max_iter' : 7500},),
        ({'loss':'log_loss', 'alpha': 0.01, 'max_iter' : 10000},),
        #({'loss':'log_loss', 'alpha': 0.05, 'max_iter' : 7500},),
        #({'loss':'log_loss', 'alpha': 0.1, 'max_iter' : 7500},),

        #({'loss':'hinge', 'alpha': 0.0001},),
        #({'loss':'hinge', 'alpha': 0.0005},),
        #({'loss':'hinge', 'alpha': 0.001},),
        #({'loss':'hinge', 'alpha': 0.005},),
        #({'loss':'hinge', 'alpha': 0.01},),
        #({'loss':'hinge', 'alpha': 0.05},),
        #({'loss':'hinge', 'alpha': 0.1},),

        #({'loss':'perceptron', 'alpha': 0.0001, 'max_iter' : 7500},),
        #({'loss':'perceptron', 'alpha': 0.0005, 'max_iter' : 7500},),
        #({'loss':'perceptron', 'alpha': 0.001, 'max_iter' : 7500},),
        #({'loss':'perceptron', 'alpha': 0.005, 'max_iter' : 7500},),
        #({'loss':'perceptron', 'alpha': 0.01, 'max_iter' : 7500},),
        #({'loss':'perceptron', 'alpha': 0.05, 'max_iter' : 7500},),
        #({'loss':'perceptron', 'alpha': 0.1, 'max_iter' : 7500},)
    ],
    #'tree':[
    #    ({},),
    #    ({'criterion': 'log_loss', 'max_depth':50},)
    #],
    'lr':[
        #({'C':1, 'solver': 'lbfgs', 'max_iter' : 7500},), #the smaller the C, the higher the regularization
        ({'C':0.5, 'solver': 'lbfgs', 'max_iter' : 10000},),
        #({'C':0.1, 'solver': 'lbfgs', 'max_iter' : 7500},),
        #({'C':0.05, 'solver': 'lbfgs', 'max_iter' : 7500},),
        #({'C':0.01, 'solver': 'lbfgs', 'max_iter' : 7500},),
        #({'C':0.005, 'solver': 'lbfgs', 'max_iter' : 7500},),

        #({'C':1, 'solver': 'liblinear', 'max_iter' : 7500},), #best for small datasets
        ({'C':0.5, 'solver': 'liblinear', 'max_iter' : 10000},),
        #({'C':0.1, 'solver': 'liblinear', 'max_iter' : 7500},),
        #({'C':0.05, 'solver': 'liblinear', 'max_iter' : 7500},),
        #({'C':0.01, 'solver': 'liblinear', 'max_iter' : 7500},),
        #({'C':0.005, 'solver': 'liblinear', 'max_iter' : 7500},),
    ],
    #'perceptron':[
    #    ({},)
    #],
    #'ridge':[
    #    ({'alpha': 1, 'max_iter' : 7500},), #the larger the stronger 1/2C
    #    ({'alpha': 1/4, 'max_iter' : 7500},),
    #    ({'alpha': 1/8, 'max_iter': 7500},),
    #    ({'alpha': 0.05, 'max_iter' : 7500},),
    #    ({'alpha': 0.01, 'max_iter' : 7500},),
    #]
}

for key in model_dict.keys():
    for i, model_info in enumerate(model_dict[key]):
        aux = (model_info[0], train_and_eval(key, model_info[0]))
        model_dict[key][i] = aux

In [699]:
for model in model_dict.keys():
    for comb in model_dict[model]:
        print('model:{} params: {}'.format(model, comb[0]))
        print(comb[1])

model:svc params: {'C': 0.1, 'max_iter': 10000}
              precision    recall  f1-score   support

           N       0.61      0.77      0.68       219
         NEU       0.22      0.06      0.09        69
        NONE       0.38      0.21      0.27        62
           P       0.61      0.68      0.64       156

    accuracy                           0.58       506
   macro avg       0.45      0.43      0.42       506
weighted avg       0.53      0.58      0.54       506

model:sgd params: {'loss': 'log_loss', 'alpha': 0.01, 'max_iter': 10000}
              precision    recall  f1-score   support

           N       0.58      0.75      0.65       219
         NEU       0.29      0.03      0.05        69
        NONE       0.37      0.18      0.24        62
           P       0.56      0.66      0.60       156

    accuracy                           0.55       506
   macro avg       0.45      0.40      0.39       506
weighted avg       0.51      0.55      0.50       506

model:lr 

In [700]:
def dummy_class():
    res = []
    for count in dev_pol_mat:
        pos = count[0]
        neg = count[1]
        if pos > neg:
            res.append('P')
            continue
        if pos < neg: 
            res.append('N')
            continue
        if pos == neg == 0:
            res.append('NONE')
            continue
        if pos == neg:
            res.append('NEU')
            continue
    return res

In [701]:
print(dev_pol_mat.shape)

(506, 2)


In [702]:
dummy_res = dummy_class()
print(classification_report(dev_df['label'], dummy_res))

              precision    recall  f1-score   support

           N       0.61      0.53      0.57       219
         NEU       0.15      0.19      0.17        69
        NONE       0.32      0.18      0.23        62
           P       0.51      0.65      0.57       156

    accuracy                           0.47       506
   macro avg       0.40      0.38      0.38       506
weighted avg       0.48      0.47      0.47       506



**Test output generation**

In [703]:
best_model = svm.LinearSVC(C = 0.1, max_iter = 10000)
train_dev_matrix = scipy.sparse.vstack((train_matrix, dev_matrix))
train_dev_labels = pd.concat([train_df['label'], dev_df['label']])
print(train_dev_matrix.shape)
print(train_dev_labels.shape)
best_model = best_model.fit(train_dev_matrix, train_dev_labels)
prediction = best_model.predict(test_matrix)

(1514, 43627)
(1514,)


In [671]:
with open('MiguelGarciaSVC.txt', 'w') as file:
    for i in range(len(prediction)):
        file.write('{}\t{}\n'.format(test_df['id'][i], prediction[i]))